# Step 2: Baseline attrition model (standard approach, no toolkit yet)

**Why a baseline?** Before trying the toolkit, we need a fair "normal" result to compare against. Later, every toolkit feature has to beat these numbers to be worth using.

**The plan (logic first):**
1. **Prepare data**: make a 0/1 target, drop useless columns.
2. **Sort columns** into categories (text) and numbers.
3. **Stratified split**: 80% train, 20% test, both keeping the same ~16% attrition rate.
4. **Model A: Logistic Regression**: simple and explainable. Needs log transform for skewed columns, scaling, one-hot encoding, and class weights for imbalance.
5. **Model B: LightGBM**: tree-based, handles skew on its own. Just one-hot encoding and class weights.
6. **Evaluate** with ROC-AUC, PR-AUC and recall (not accuracy, because of imbalance).
7. **Cross-validation**: the test set has only ~50 leavers, so one split can be lucky or unlucky. 5-fold CV gives a more stable number.
8. **Save results** so we can compare toolkit versions later.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, confusion_matrix
from lightgbm import LGBMClassifier

RANDOM_STATE = 42

## 1. Prepare data
Target: `left` = 1 if Attrition is "Yes". Then drop the original Attrition column (otherwise the model would just copy the answer) and the four useless columns we found in the EDA.

In [2]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

y = (df["Attrition"] == "Yes").astype(int)
X = df.drop(columns=["Attrition", "EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"])

print("X shape:", X.shape)
print("Attrition rate:", round(y.mean(), 3))

X shape: (1470, 30)
Attrition rate: 0.161


## 2. Sort columns into types
- **Categorical**: text columns like JobRole, Department, OverTime → one-hot encode.
- **Skewed numeric**: income and tenure columns → log transform (for logistic regression only).
- **Other numeric**: everything else.

In [3]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
skewed_cols = ["MonthlyIncome", "YearsAtCompany", "TotalWorkingYears", "YearsSinceLastPromotion"]
num_cols = [c for c in X.columns if c not in cat_cols + skewed_cols]

print("Categorical:", cat_cols)
print("Skewed numeric:", skewed_cols)
print("Other numeric:", len(num_cols), "columns")

Categorical: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Skewed numeric: ['MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'YearsSinceLastPromotion']
Other numeric: 19 columns


## 3. Stratified train/test split
`stratify=y` keeps the attrition rate the same in train and test.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "attrition rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "attrition rate:", round(y_test.mean(), 3))
print("Leavers in test set:", y_test.sum())

Train: (1176, 30) attrition rate: 0.162
Test:  (294, 30) attrition rate: 0.16
Leavers in test set: 47


## 4. Model A: Logistic Regression pipeline
A **Pipeline** bundles preprocessing + model, so the same steps are applied to train and test (and inside cross-validation) with no leakage.
- Skewed columns: `log1p` then scale
- Other numbers: scale
- Categories: one-hot
- `class_weight="balanced"`: makes mistakes on leavers count more, so the model doesn't ignore the small class.

In [5]:
lr_preprocess = ColumnTransformer([
    ("skewed", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("scale", StandardScaler())]), skewed_cols),
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

lr_model = Pipeline([
    ("prep", lr_preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000)),
])

## 5. Model B: LightGBM pipeline
Trees split on thresholds, so skew and scaling don't matter. We only one-hot the categories.

In [6]:
lgbm_preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

lgbm_model = Pipeline([
    ("prep", lgbm_preprocess),
    ("model", LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=15,
        class_weight="balanced", random_state=RANDOM_STATE, verbose=-1,
    )),
])

## 6. Train and evaluate on the test set
- **ROC-AUC**: how well the model ranks leavers above stayers (0.5 = random, 1.0 = perfect).
- **PR-AUC**: focuses on the small class. Random would be ~0.16 here, so anything well above that is useful.
- **Recall**: of the people who actually left, how many did we catch?
- **Precision**: of the people we flagged, how many actually left?

In [7]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    print(f"=== {name} ===")
    print("Confusion matrix [[TN FP] [FN TP]]:")
    print(confusion_matrix(y_test, pred), "\n")

    return {
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "recall": recall_score(y_test, pred),
        "precision": precision_score(y_test, pred),
    }

results = [
    evaluate("Baseline LogReg", lr_model),
    evaluate("Baseline LightGBM", lgbm_model),
]

results_df = pd.DataFrame(results).round(3)
results_df

=== Baseline LogReg ===
Confusion matrix [[TN FP] [FN TP]]:
[[199  48]
 [ 16  31]] 

=== Baseline LightGBM ===
Confusion matrix [[TN FP] [FN TP]]:
[[229  18]
 [ 26  21]] 



,model,roc_auc,pr_auc,recall,precision
0,Baseline LogReg,0.802,0.561,0.660,0.392
1,Baseline LightGBM,0.780,0.510,0.447,0.538


## 7. Cross-validation (more stable estimate)
Split the training data 5 ways, train on 4 parts and test on the 5th, repeat 5 times. The **mean** tells us typical performance; the **std** tells us how much it jumps around.

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, model in [("Baseline LogReg", lr_model), ("Baseline LightGBM", lgbm_model)]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
    cv_rows.append({"model": name, "cv_auc_mean": scores.mean(), "cv_auc_std": scores.std()})

cv_df = pd.DataFrame(cv_rows).round(3)
cv_df

,model,cv_auc_mean,cv_auc_std
0,Baseline LogReg,0.833,0.029
1,Baseline LightGBM,0.810,0.030


## 8. Quick look: what drives the Logistic Regression?
Positive coefficient = pushes toward leaving. Negative = pushes toward staying. A sanity check that the model learned sensible things (OverTime should be near the top).

In [9]:
feature_names = lr_model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr_model.named_steps["model"].coef_[0], index=feature_names).sort_values()

print("Top 8 pushing toward LEAVING:")
print(coefs.tail(8)[::-1].round(2), "\n")
print("Top 8 pushing toward STAYING:")
print(coefs.head(8).round(2))

Top 8 pushing toward LEAVING:
cat__JobRole_Laboratory Technician       1.02
cat__JobRole_Sales Representative        0.90
cat__BusinessTravel_Travel_Frequently    0.84
cat__OverTime_Yes                        0.79
num__JobLevel                            0.78
cat__EducationField_Human Resources      0.72
num__NumCompaniesWorked                  0.54
cat__JobRole_Human Resources             0.50
dtype: float64 

Top 8 pushing toward STAYING:
cat__JobRole_Research Director   -1.53
cat__BusinessTravel_Non-Travel   -1.02
cat__OverTime_No                 -0.88
skewed__TotalWorkingYears        -0.68
cat__EducationField_Other        -0.68
skewed__MonthlyIncome            -0.54
num__EnvironmentSatisfaction     -0.45
cat__MaritalStatus_Divorced      -0.43
dtype: float64


## 9. Save results for later comparison
When we add toolkit features in the next notebook, we'll add new rows to this file and compare.

In [10]:
summary = results_df.merge(cv_df, on="model")
summary.to_csv("results.csv", index=False)
summary

,model,roc_auc,pr_auc,recall,precision,cv_auc_mean,cv_auc_std
0,Baseline LogReg,0.802,0.561,0.660,0.392,0.833,0.029
1,Baseline LightGBM,0.780,0.510,0.447,0.538,0.810,0.030
